# 04 · The SO-101 is a device

**What you will see**

- The arm's **manifest**, generated from its description: sensors, controls (with `range`, `safe`, `on_disconnect`), rpcs and events, next to its G1 neighbour's.
- A write outside a joint's range, **refused** with a message naming the joint, the value and the bound.
- The **safety chain** order (range → safe → interlocks → validators → operator gate), shown by tracing it.
- **`estop()` from another thread** in the middle of a run: the arm holds, and the run returns `stopped_by='estop'`.
- `resume()` refused without `approved_by`, then released by a named operator; `hold()`.
- `subscribe('state', cb, hz=)` samples and their measured rate.
- The **watchdog** applying `on_disconnect` when the heartbeat stops.

Everything runs in MuJoCo on this laptop; no robot needed.

In [1]:
import _nb
_nb.drawing("d06_device")

## One world, one arm, a G1 next door

In [2]:
import threading, time
import numpy as np
from PIL import Image, ImageDraw
from strands_robots import Robot, Simulation
from strands_robots.robot.safety import RequestRefused, EStopped, Watchdog

sim = Simulation("mujoco", render_size=(480, 360))
arm = Robot("so101", sim=sim)
sim.camera("cam", position=(0.55, -0.45, 0.4), target=(0.0, 0.0, 0.15), size=(480, 360))
g1 = Robot("g1", sim=Simulation())          # the neighbour, in its own world (we only read its manifest here)
print(arm.id, "in", arm.sim.backend_name, "|", g1.id, "in", g1.sim.backend_name)

so101 in mujoco | unitree_g1 in mujoco


## The manifest

`robot.manifest` is generated from the description and the body, never written by hand. `describe()` is the same thing as JSON, plus the live `task` and `estop`.

In [3]:
_nb.show(_nb.table([
    {"": k, "so101": getattr(arm.manifest, k), "unitree_g1": getattr(g1.manifest, k)}
    for k in ("device_id", "device_type", "name", "real", "position_hold", "world", "tenant")
] + [
    {"": "sensors", "so101": len(arm.manifest.sensors), "unitree_g1": len(g1.manifest.sensors)},
    {"": "controls", "so101": len(arm.manifest.controls), "unitree_g1": len(g1.manifest.controls)},
]))
print("G1 has a floating base, so it adds a pose sensor:", g1.manifest.sensors[-1])

|  | so101 | unitree_g1 |
|---|---|---|
| device_id | so101 | unitree_g1 |
| device_type | robot_arm | humanoid |
| name | so101 | unitree_g1 |
| real | False | False |
| position_hold | True | True |
| world | mujoco | mujoco |
| tenant | default | default |
| sensors | 18 | 88 |
| controls | 6 | 29 |

G1 has a floating base, so it adds a pose sensor: Sensor(name='base.pose', unit='m,quat', kind='pose')


In [4]:
fmt = lambda b: "—" if b is None else f"[{b[0]:+.4f}, {b[1]:+.4f}]"
_nb.table([
    {"control": c.name, "unit": c.unit, "range": fmt(c.range), "safe": fmt(c.safe), "on_disconnect": c.on_disconnect}
    for c in arm.manifest.controls
])

| control | unit | range | safe | on_disconnect |
|---|---|---|---|---|
| shoulder_pan.pos | rad | [-1.9199, +1.9199] | [-1.9199, +1.9199] | hold |
| shoulder_lift.pos | rad | [-1.7453, +1.7453] | [-1.7453, +1.7453] | hold |
| elbow_flex.pos | rad | [-1.7453, +1.5708] | [-1.7453, +1.5708] | hold |
| wrist_flex.pos | rad | [-1.6581, +1.6581] | [-1.6581, +1.6581] | hold |
| wrist_roll.pos | rad | [-2.7925, +2.7925] | [-2.7925, +2.7925] | hold |
| gripper.pos | rad | [-0.1745, +1.7453] | [-0.1745, +1.7453] | hold |

In [5]:
_nb.table([
    {"rpc": r.name, "params": ", ".join(
        f"{k}{'*' if k in r.params.get('required', []) else ''}: {v.get('type', 'any')}"
        for k, v in r.params.get("properties", {}).items()) or "—", "doc": r.doc}
    for r in arm.manifest.rpcs
])

| rpc | params | doc |
|---|---|---|
| describe | — | The manifest as JSON, plus the live ``task`` and ``estop`` status. |
| observe | images: boolean |  |
| act | action*: any, steps: integer | Write targets for THIS body and advance its world one control step (unless manual time). |
| run | policy*: any, instruction: string, hz: number, seconds: number, steps: integer, stop_when: any, units: string, realtime: boolean, stop_event: any, record: string, push: boolean | observe -> policy -> act at ``hz`` (default: the policy's ``hz``, else 30) until a limit or ``stop()``. |
| stop | — |  |
| hold | — | Keep the current joint positions (under torque on hardware). |
| estop | — | Stop now: bypasses the safety chain, never waits for the world or the run loop; latches. |
| resume | approved_by: string | Release the e-stop latch; refused without the approving operator's name. |
| calibrate | interactive: boolean | Read (or, at the arm, record) the bus calibration: LeRobot's calibration JSON, unchanged. |

In [6]:
_nb.show(_nb.table([{"event": e.name, "hz": e.hz or "on change"} for e in arm.manifest.events]))
d = arm.describe()
print("describe() keys:", sorted(d)[:12], "...")
print("task:", d["task"], "| estop:", d["estop"])

| event | hz |
|---|---|
| state | 10.0 |
| fault | on change |
| estop | on change |
| run.started | on change |
| run.stopped | on change |

describe() keys: ['action_keys', 'category', 'controls', 'description', 'device_id', 'device_type', 'dof', 'estop', 'events', 'floating', 'format', 'id'] ...
task: None | estop: False


## A write outside the range

`confirm_act()` is what an agent tool or a dashboard calls. The refusal names the joint, the value and the bound.

In [7]:
try:
    arm.confirm_act({"elbow_flex": 3.1})
except RequestRefused as e:
    print(type(e).__name__, "→", e)
print("RequestRefused is a PermissionError:", issubclass(RequestRefused, PermissionError))

RequestRefused → so101: elbow_flex=3.1000 rad is outside its range [-1.7453, 1.5708]
RequestRefused is a PermissionError: True


## The safety chain, in order

| step | checks | on refusal |
|---|---|---|
| range | target inside the control's hard range | `RequestRefused` naming joint, value, bound |
| safe | inside the soft bound, or the operator's `confirm=` says yes (asked once) | `RequestRefused` |
| interlocks | `robot.interlocks`: another value must agree | the interlock's message |
| validators | `robot.validators`: pure checks `(device, request)` | the validator's message, verbatim |
| operator gate | a real body needs `confirm=`; a simulated body passes | `RequestRefused` |

Below, a tracer on each hook shows which steps ran. A range refusal stops before any interlock or validator runs; a validator refusal arrives verbatim; `robot.act()` in your own code is the operator's and skips the chain.

In [8]:
trace = []
def interlock(robot, request):
    trace.append("interlock")

def cup_held(robot, request):
    trace.append("validator")
    if abs(dict(request.items())["wrist_roll"]) > 0.1:
        raise RequestRefused("wrist_roll must stay within 0.1 rad of zero while the cup is held")

arm.interlocks.append(interlock)
arm.validators.append(cup_held)

rows = []
for label, action in [("out of range", {"elbow_flex": 3.1}),
                      ("validator says no", {"wrist_roll": 0.5}),
                      ("all clear", {"wrist_roll": 0.05})]:
    trace.clear()
    try:
        arm.confirm_act(action); outcome = "written"
    except RequestRefused as e:
        outcome = f"refused: {e}"
    ran = " → ".join(["range", "safe", *trace]) if trace else "range ✗ (nothing after it ran)"
    rows.append({"request": label, "action": action, "steps that ran": ran, "outcome": outcome})
arm.act({"wrist_roll": 0.5})   # the operator's own code: no chain
rows.append({"request": "robot.act (operator)", "action": {"wrist_roll": 0.5}, "steps that ran": "none", "outcome": f"written, wrist_roll={arm.observe().joints['wrist_roll']:+.3f}"})
arm.interlocks.remove(interlock); arm.validators.remove(cup_held)
_nb.table(rows)

| request | action | steps that ran | outcome |
|---|---|---|---|
| out of range | {'elbow_flex': 3.1} | range ✗ (nothing after it ran) | refused: so101: elbow_flex=3.1000 rad is outside its range [-1.7453, 1.5708] |
| validator says no | {'wrist_roll': 0.5} | range → safe → interlock → validator | refused: wrist_roll must stay within 0.1 rad of zero while the cup is held |
| all clear | {'wrist_roll': 0.05} | range → safe → interlock → validator | written |
| robot.act (operator) | {'wrist_roll': 0.5} | none | written, wrist_roll=+0.035 |

## `estop()` from another thread, mid-run

A policy waves the arm in real time. After 2 s another thread calls `arm.estop()`: no chain, no lock, it latches. We then let the world run 2 s more to watch the arm hold.

In [9]:
arm.act({k: 0.0 for k in arm.action_keys}, steps=50)
frames, when = [], {}

def caption(img, text):
    im = Image.fromarray(img); d = ImageDraw.Draw(im)
    d.rectangle([0, 0, im.width, 22], fill=(20, 20, 20)); d.text((8, 5), text, fill=(240, 240, 240))
    return np.asarray(im)

def policy(obs):
    t = obs.t
    return {"shoulder_pan": 0.8 * np.sin(1.5 * t), "shoulder_lift": -0.4 + 0.3 * np.sin(t),
            "elbow_flex": 0.6 + 0.4 * np.sin(1.2 * t), "wrist_flex": 0.5 * np.sin(2 * t), "gripper": 0.6}

def film(obs):                       # stop_when hook: runs on this thread, so rendering is safe
    if len(frames) % 1 == 0 and int(round(obs.t * 50)) % 3 == 0:
        frames.append(caption(sim.render("cam"), f"arm.run(policy)   t={obs.t:4.1f}s   running"))
    return False

def press():
    time.sleep(2.0)
    when["estop"] = time.perf_counter()
    arm.estop()

threading.Thread(target=press, name="operator").start()
res = arm.run(policy, hz=50, seconds=10, realtime=True, stop_when=film)
when["returned"] = time.perf_counter()

q0 = np.array(arm.observe().joints.positions); drift = 0.0
for _ in range(30):                   # 2 s more world time with the latch engaged
    sim.step(int((2 / 30) / sim.physics.timestep))
    drift = max(drift, float(np.max(np.abs(np.array(arm.observe().joints.positions) - q0))))
    frames.append(caption(sim.render("cam"), f"estop()   t={sim.time:4.1f}s   holding, drift {np.rad2deg(drift):.2f} deg"))

print({k: getattr(res, k) for k in ("steps", "seconds", "hz_target", "stopped_by", "fault")})
print(f"stopped_by={res.stopped_by!r} after {res.steps} steps; estop() → run returned in "
      f"{(when['returned'] - when['estop']) * 1000:.1f} ms; max drift while holding {np.rad2deg(drift):.2f}°")
assert res.stopped_by == "estop"
_nb.video(frames, fps=15)

{'steps': 100, 'seconds': 2.0, 'hz_target': 50.0, 'stopped_by': 'estop', 'fault': None}
stopped_by='estop' after 100 steps; estop() → run returned in 2.3 ms; max drift while holding 1.85°


## The latch, `resume()` and `hold()`

While latched, every write is refused (even the operator's `act`). `resume()` needs the name of the operator who approves it.

In [10]:
print("estopped:", arm.estopped, "| describe()['estop']:", arm.describe()["estop"])
for label, call in [("act while latched", lambda: arm.act({"shoulder_pan": 0.2})),
                    ("resume()", lambda: arm.resume()),
                    ("resume(approved_by='  ')", lambda: arm.resume(approved_by="  "))]:
    try:
        call(); print(f"{label}: accepted (unexpected)")
    except RequestRefused as e:
        print(f"{label}: refused ({type(e).__name__}) → {e}")
print("still estopped:", arm.estopped, "| EStopped is a RequestRefused:", issubclass(EStopped, RequestRefused))
arm.resume(approved_by="alice")
print("resume(approved_by='alice') → estopped:", arm.estopped)
arm.act({"shoulder_pan": 0.3}, steps=50)
print("act after resume: shoulder_pan =", round(arm.observe().joints["shoulder_pan"], 3))

estopped: True | describe()['estop']: True
act while latched: refused (EStopped) → so101: e-stop is engaged (estop()); resume(approved_by=<operator>) first
resume(): refused (RequestRefused) → so101: resume needs approved_by=<the operator who cleared the stop>
resume(approved_by='  '): refused (RequestRefused) → so101: resume needs approved_by=<the operator who cleared the stop>
still estopped: True | EStopped is a RequestRefused: True
resume(approved_by='alice') → estopped: False
act after resume: shoulder_pan = 0.259


In [11]:
arm.act({"shoulder_pan": 0.6, "elbow_flex": 0.8}, steps=100)
arm.hold()                                     # keep the current positions
q0 = np.array(arm.observe().joints.positions)
sim.step(int(1.0 / sim.physics.timestep))
q1 = np.array(arm.observe().joints.positions)
print(f"hold(): max joint change over 1 s of world time {np.rad2deg(np.max(np.abs(q1 - q0))):.3f}°")

hold(): max joint change over 1 s of world time 1.483°


## `subscribe('state', cb, hz=)`

A background run moves the arm while we listen. The payload is the joints dict a policy sees (`<label>.pos/.vel/.effort`) plus `t` (monotonic) and `seq`.

In [12]:
got = []
unsubscribe = arm.subscribe("state", got.append, hz=50)
task = arm.start(policy, hz=50, seconds=1.2, realtime=True)   # something to watch
time.sleep(1.0)
unsubscribe()
task.halt(2.0)
n = len(got); time.sleep(0.1)
rate = (n - 1) / (got[-1]["t"] - got[0]["t"])
_nb.show(_nb.table([{"seq": g["seq"], "t − t₀ (s)": f"{g['t'] - got[0]['t']:.4f}", **{k: f"{g[k]:+.4f}" for k in ("shoulder_pan.pos", "elbow_flex.pos", "gripper.pos")}} for g in got[::10]]))
print(f"{n} samples in ~1 s, asked hz=50, measured {rate:.1f} Hz; seq contiguous: {[g['seq'] for g in got] == list(range(n))}; "
      f"silent after unsubscribe: {len(got) == n}")

| seq | t − t₀ (s) | shoulder_pan.pos | elbow_flex.pos | gripper.pos |
|---|---|---|---|---|
| 0 | 0.0000 | +0.6542 | +0.8287 | +0.5995 |
| 10 | 0.2034 | +0.7023 | +0.7618 | +0.5996 |
| 20 | 0.4034 | +0.5559 | +0.8738 | +0.5996 |
| 30 | 0.6036 | +0.3577 | +0.9280 | +0.5996 |
| 40 | 0.8039 | +0.1061 | +0.9860 | +0.5997 |
| 50 | 1.0027 | -0.1338 | +1.0119 | +0.5997 |

51 samples in ~1 s, asked hz=50, measured 49.9 Hz; seq contiguous: True; silent after unsubscribe: True


## The watchdog and `on_disconnect`

`Watchdog(robot, timeout_ms=)` expects a `beat()` on every command. When the beats stop for longer than the timeout, each control applies its `on_disconnect` (here `hold`), once. On a simulated body its clock is world time, so we step the world by hand to show the exact boundary.

In [13]:
arm.act({"shoulder_pan": 0.6}, steps=50)
rows = []
with sim.manual_time():
    dog = Watchdog(arm, timeout_ms=100)
    dog.beat(); t0 = sim.time
    rows.append({"world ms since beat": 0, "poll()": "—", "note": "beat()"})
    sim.step(int(0.05 / sim.physics.timestep))
    rows.append({"world ms since beat": round((sim.time - t0) * 1000), "poll()": dog.poll(), "note": "within timeout"})
    sim.step(int(0.06 / sim.physics.timestep))
    fired = dog.poll()
    q = arm.observe().joints.positions
    held = np.allclose(sim.backend.data.ctrl[: len(q)], q)
    rows.append({"world ms since beat": round((sim.time - t0) * 1000), "poll()": fired, "note": f"lapsed → on_disconnect='hold' applied (ctrl == current q: {held})"})
    rows.append({"world ms since beat": round((sim.time - t0) * 1000), "poll()": dog.poll(), "note": "applied once per lapse"})
    dog.beat()
    rows.append({"world ms since beat": 0, "poll()": dog.poll(), "note": "beat() again re-arms it"})
_nb.table(rows)

| world ms since beat | poll() | note |
|---|---|---|
| 0 | — | beat() |
| 50 | False | within timeout |
| 110 | True | lapsed → on_disconnect='hold' applied (ctrl == current q: True) |
| 110 | False | applied once per lapse |
| 0 | False | beat() again re-arms it |

In [14]:
from strands_robots.robot.device import Control, ON_DISCONNECT
print("on_disconnect is one of", ON_DISCONNECT)
for kw in ({"on_disconnect": "apply"}, {"safe": (-3.0, 0.0)}):
    try:
        Control("gripper.pos", "rad", (-0.17, 1.75), kw.get("safe", (-0.17, 1.75)), kw.get("on_disconnect", "hold"))
    except ValueError as e:
        print("refused at build time:", e)
print(Control("gripper.pos", "rad", (-0.17, 1.75), (-0.17, 1.75), "apply", disconnect_value=0.0))

on_disconnect is one of ('hold', 'zero', 'safe', 'apply')
refused at build time: gripper.pos: on_disconnect='apply' needs a disconnect_value
refused at build time: gripper.pos: safe [-3.0, 0.0] is not inside range [-0.17, 1.75]
Control(name='gripper.pos', unit='rad', range=(-0.17, 1.75), safe=(-0.17, 1.75), on_disconnect='apply', disconnect_value=0.0, interlocks=())


In [15]:
g1.sim.close(); sim.close()
print("closed")

closed


## Where this sits in v1

A `Robot` is a device. Its manifest is the one surface the mesh (`05_mesh`), the agent tools and the dashboard read, so they never describe the arm themselves. Every write from an agent or a dashboard passes the same safety chain; `estop()` bypasses it, latches, and only a named operator releases it. The physical SO-101 gets the same manifest with `real=True` and the operator gate switched on; the e-stop on the bench arm is the one row still waiting for hardware.